# Lab 4 — Framework Quickstart (Keras & PyTorch)

**Note:** This notebook is optional for the first lab but helpful as a bridge to future sessions.

You will:  
- Train a small classifier in **Keras**  
- Train the same idea in **PyTorch**

## 0) Install (if needed)

Uncomment what you need below and run once in your environment.

In [1]:
# !pip install tensorflow
# !pip install torch torchvision torchaudio

## 1) Make a Simple Dataset (same two blobs as before)

In [2]:
import numpy as np

def set_seed(seed=42): np.random.seed(seed)
set_seed(3)

def make_blobs(n_per_class=300, d=2, gap=1.5):
    mean0 = -gap*np.ones(d)
    mean1 =  gap*np.ones(d)
    X0 = np.random.randn(n_per_class, d) + mean0
    X1 = np.random.randn(n_per_class, d) + mean1
    X = np.vstack([X0, X1]).astype("float32")
    y = np.hstack([np.zeros(n_per_class), np.ones(n_per_class)]).astype("float32")
    idx = np.random.permutation(len(X))
    return X[idx], y[idx]

X, y = make_blobs()
split = int(0.8*len(X))
X_tr, y_tr = X[:split], y[:split]
X_va, y_va = X[split:], y[split:]
print(X_tr.shape, X_va.shape)

(480, 2) (120, 2)


## 2) Keras (TensorFlow)

In [3]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    tf.random.set_seed(42)

    model = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-2),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=25, verbose=0)
    print("Keras val accuracy:", model.evaluate(X_va, y_va, verbose=0)[1])
except Exception as e:
    print("Keras/TensorFlow not available in this environment:", e)

2025-09-05 16:22:09.896465: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1757103732.373004   19232 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4080 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 Ti, pci bus id: 0000:01:00.0, compute capability: 7.5
2025-09-05 16:22:13.933177: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fd6b8004170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-05 16:22:13.933215: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce GTX 1660 Ti, Compute Capability 7.5
2025-09-05 16:22:13.952096: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabl

Keras val accuracy: 0.9666666388511658


## 3) PyTorch

In [4]:
try:
    import torch
    from torch import nn
    from torch.utils.data import TensorDataset, DataLoader

    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_tr_t = torch.tensor(X_tr).to(device)
    y_tr_t = torch.tensor(y_tr).to(device)
    X_va_t = torch.tensor(X_va).to(device)
    y_va_t = torch.tensor(y_va).to(device)

    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=64, shuffle=True)

    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 16),
                nn.ReLU(),
                nn.Linear(16, 1),
                nn.Sigmoid()
            )
        def forward(self, x): return self.net(x)

    model = MLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.BCELoss()

    for ep in range(25):
        model.train()
        for xb, yb in dl:
            opt.zero_grad()
            pred = model(xb).squeeze(1)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(X_va_t).squeeze(1)
        acc = ((pred >= 0.5) == y_va_t.bool()).float().mean().item()
    print("PyTorch val accuracy:", acc)
except Exception as e:
    print("PyTorch not available in this environment:", e)

PyTorch val accuracy: 0.9750000238418579


### Additional Steps to try
- Save trained models and load them later  
- Try different activations, hidden sizes, or optimizers  
- Use learning rate scheduling